# Day ahead capacity market study

In [1]:
# Imports for pythonifying code

from typing import Callable, List, Dict, Any

# Path related imports

import os
import sys

In [2]:
# Append utilities path 

current_path = os.getcwd()
cwd = os.path.dirname(current_path)

sys.path.append(os.path.join(cwd, "utilities"))

## Get input data

In [3]:
# Import pickle operations 

from pickle_file_operations import read_pickle, write_pickle

# Import dataclasses

from optimization_dataclasses import Prosumer_Cluster_Data, Market_Data, Bivariate_Pwl_Input, Layer_Data, Optimization_Results
from dataclasses import fields

## Get neural network data

In [4]:
# Relevant imports

import tensorflow as tf
from keras import layers
import numpy as np
import pandas as pd

In [5]:
def get_layer_data(layer: layers) -> Layer_Data:
    
    input_size, output_size = np.shape(layer.get_weights()[0])
        
    coefficient_matrix, bias_matrix = layer.get_weights()
    
    bias_matrix = np.reshape(bias_matrix, (1, len(bias_matrix)))
    
    activation_type = layer.activation.__name__
    
    return Layer_Data(activation_type=activation_type,
                      input_size=input_size,
                      output_size=output_size, 
                      coefficient_matrix=coefficient_matrix, 
                      bias_matrix=bias_matrix)

## Create the optimization model

In [6]:
# Import gurobi API and relevant functions

import gurobipy as gp
from gurobipy import GRB
from gurobi_ml import add_predictor_constr
import time as Time

In [7]:
def _add_market_variables(opt_model: gp.Model, time_steps: int):
    
    variable_names = ["flexibility_MWh", "updated_max_flexibility_MWh", "flexibility_cost_DKK"]
    lower_bounds = [0.098, 0.10, 0]
    upper_bounds = [GRB.INFINITY for i in range(3)]
    decision_variables = {}
    
    for time in range(time_steps):
        
        decision_variables[time] = {variable_names[i]: opt_model.addVar(vtype=GRB.CONTINUOUS, 
                                                                        lb=lower_bounds[i], ub=upper_bounds[i],
                                                                       name=variable_names[i]+"_"+str(time)) 
                                    for i in range(3)} 
        
    opt_model.update()
    
    return decision_variables

def _add_market_constraints(opt_model: gp.Model, decision_variables: Dict[str, Any],  prosumer_data: Prosumer_Cluster_Data, 
                            activation_signal: np.ndarray, epsilon: float):
    
    cross_temporal_matrix = prosumer_data.cross_temporal_matrix
    maximum_flexibility_profile = prosumer_data.max_flexibility_profile 
    time_steps = len(maximum_flexibility_profile)
    
    for time in range(time_steps):
        variables = decision_variables[time]
        
        opt_model.addConstr(variables["flexibility_MWh"] + epsilon <= variables["updated_max_flexibility_MWh"])
        opt_model.addConstr(variables["updated_max_flexibility_MWh"] 
                            == maximum_flexibility_profile[time] 
                            - (gp.quicksum(cross_temporal_matrix[j][time] * decision_variables[j]["flexibility_MWh"] 
                                           * activation_signal[j]
                                          for j in range(time_steps)
                                          )
                              )
                           )
    opt_model.update()
    
    return None


In [8]:
def _add_pwl_variables(opt_model: gp.Model, pwl_input: Bivariate_Pwl_Input, time_steps: int):

    # Add decion variables associated with the PWL constraints
    pwl_variables = {}
    
    for time in range(time_steps):

        # Break points for PWL
        flex_points = pwl_input[time].flexibility_points
        max_flex_points = pwl_input[time].max_flexibility_points
        
        # Number of segments
        num_segments_x = len(flex_points) - 1
        num_segments_y = len(max_flex_points) - 1

        # PWL variabless
        pwl_variables[time] = {"weights": opt_model.addVars(num_segments_x + 1, num_segments_y + 1, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0,
                                                           name=str(time) + "_weights"), 
                               "segment_activation_x": opt_model.addVars(num_segments_x, vtype=GRB.BINARY, lb=0.0, ub=1.0, 
                                                                         name=str(time) + "_segment_activation_x"), 
                               "segment_activation_y": opt_model.addVars(num_segments_y, vtype=GRB.BINARY, lb=0.0, ub=1.0, 
                                                                        name=str(time) + "_segment_activation_y"), 
                               "simplex_selection": opt_model.addVar(vtype=GRB.BINARY, lb=0.0, ub=1.0, name=str(time) + "_simplex_selection")}
    opt_model.update()

    return pwl_variables
    
def _add_pwl_constraints(opt_model: gp.Model, market_variables: Dict[str, Any], pwl_variables: Dict[str, Any], 
                         pwl_input: Dict[int, Bivariate_Pwl_Input], time_steps: int):
    
    # PWL constraints

    for time in range(time_steps):

        # Break points for PWL
        flex_points = pwl_input[time].flexibility_points
        max_flex_points = pwl_input[time].max_flexibility_points
        cost_points = pwl_input[time].cost_points
    
        # Number of segments
        num_segments_x = len(flex_points) - 1
        num_segments_y = len(max_flex_points) - 1
    
        # Size of the segments
        delta_x = flex_points[1] - flex_points[0]
        delta_y = max_flex_points[1] - max_flex_points[0]
    
    
        pwl_decision_variables = pwl_variables[time]
        market_decision_variables = market_variables[time]
    # Box constraints

    # Select only one segment for x-value and y-vale

        opt_model.addConstr(1 
                            == gp.quicksum(pwl_decision_variables["segment_activation_x"][i] 
                                        for i in range(num_segments_x))
                           )
        
        opt_model.addConstr(1 == gp.quicksum(pwl_decision_variables["segment_activation_y"][j] 
                                        for j in range(num_segments_y))
                           )
    # Sum of all weights must be equal to 1

        opt_model.addConstr(1 == 
                            gp.quicksum(pwl_decision_variables["weights"][i, j]
                                       for i in range(num_segments_x + 1)
                                       for j in range(num_segments_y + 1))
                           )
    # Set weights of unselected segments as zero 

    # For x-values

        opt_model.addConstr(gp.quicksum(pwl_decision_variables["weights"][0, j]
                                       for j in range(num_segments_y + 1)) 
                            <= pwl_decision_variables["segment_activation_x"][0]
                           )

        opt_model.addConstr(gp.quicksum(pwl_decision_variables["weights"][num_segments_x, j]
                                       for j in range(num_segments_y + 1)) 
                            <= pwl_decision_variables["segment_activation_x"][num_segments_x - 1]
                           )

        opt_model.addConstrs(gp.quicksum(pwl_decision_variables["weights"][i, j]
                                       for j in range(num_segments_y + 1)) 
                             <= pwl_decision_variables["segment_activation_x"][i-1] 
                             + pwl_decision_variables["segment_activation_x"][i]
                             for i in range(1, num_segments_x)
                            )

    # For y-values
        opt_model.addConstr(gp.quicksum(pwl_decision_variables["weights"][i, 0]
                                       for i in range(num_segments_x + 1)) 
                            <= pwl_decision_variables["segment_activation_y"][0]
                           )

        opt_model.addConstr(gp.quicksum(pwl_decision_variables["weights"][i, num_segments_y]
                                       for i in range(num_segments_x + 1)) 
                            <= pwl_decision_variables["segment_activation_y"][num_segments_x - 1]
                           )

        opt_model.addConstrs(gp.quicksum(pwl_decision_variables["weights"][i, j]
                                       for i in range(num_segments_x + 1)) 
                             <= pwl_decision_variables["segment_activation_y"][j-1] 
                             + pwl_decision_variables["segment_activation_y"][j]
                             for j in range(1, num_segments_y)
                            )   

    # Simplex constraints

    # Select the relevant simplex

        opt_model.addConstr(-1 * delta_x * delta_y * (1 - pwl_decision_variables["simplex_selection"]) 
                            <= delta_y * (market_decision_variables["flexibility_MWh"] 
                                          - gp.quicksum(pwl_decision_variables["segment_activation_x"][i] * flex_points[i] 
                                                        for i in range(num_segments_x)
                                                       )
                                         ) 
                            - delta_x * (market_decision_variables["updated_max_flexibility_MWh"] 
                                         - gp.quicksum(pwl_decision_variables["segment_activation_y"][j] * max_flex_points[j] 
                                                       for j in range(num_segments_y)
                                                      )
                                        )
                           )
        
        opt_model.addConstr(delta_x * delta_y * (1 - pwl_decision_variables["simplex_selection"]) 
                            >= delta_y * (market_decision_variables["flexibility_MWh"] 
                                          - gp.quicksum(pwl_decision_variables["segment_activation_x"][i] * flex_points[i] 
                                                        for i in range(num_segments_x)
                                                       )
                                         ) 
                            - delta_x * (market_decision_variables["updated_max_flexibility_MWh"] 
                                         - gp.quicksum(pwl_decision_variables["segment_activation_y"][j] * max_flex_points[j] 
                                                       for j in range(num_segments_y)
                                                      )
                                        )
                           )
    # Select the relevant vertices

    # General case

        opt_model.addConstrs(pwl_decision_variables["weights"][i+1, j] 
                             <= 2 + pwl_decision_variables["simplex_selection"] 
                             - pwl_decision_variables["segment_activation_x"][i] 
                             - pwl_decision_variables["segment_activation_y"][j] 
                             for i in range(num_segments_x - 1) 
                             for j in range(num_segments_y)
                            )

        opt_model.addConstrs(pwl_decision_variables["weights"][i, j+1] 
                             <= 3 - pwl_decision_variables["simplex_selection"] 
                             - pwl_decision_variables["segment_activation_x"][i] 
                             - pwl_decision_variables["segment_activation_y"][j] 
                             for i in range(num_segments_x) 
                             for j in range(num_segments_y - 1)
                            )
    # Edge case

        opt_model.addConstrs(pwl_decision_variables["weights"][num_segments_x, j] 
                             <= 2 + pwl_decision_variables["simplex_selection"] 
                             - pwl_decision_variables["segment_activation_x"][num_segments_x-1] 
                             - pwl_decision_variables["segment_activation_y"][j]
                            for j in range(num_segments_y)
                            )

        opt_model.addConstrs(pwl_decision_variables["weights"][i, num_segments_y] 
                             <= 3 - pwl_decision_variables["simplex_selection"] 
                             - pwl_decision_variables["segment_activation_x"][i] 
                             - pwl_decision_variables["segment_activation_y"][num_segments_y-1]
                            for i in range(num_segments_x)
                            )
   # Calculate x-value, y-value and z-value

        opt_model.addConstr(market_decision_variables["flexibility_MWh"] 
                            == gp.quicksum(pwl_decision_variables["weights"][i, j] * flex_points[i] 
                                           for i in range(num_segments_x+1) 
                                           for j in range(num_segments_y+1))
                           )

        opt_model.addConstr(market_decision_variables["updated_max_flexibility_MWh"] 
                            == gp.quicksum(pwl_decision_variables["weights"][i, j] * max_flex_points[j] 
                                           for i in range(num_segments_x+1) 
                                           for j in range(num_segments_y+1))
                           )

        opt_model.addConstr(market_decision_variables["flexibility_cost_DKK"] 
                            == gp.quicksum(pwl_decision_variables["weights"][i, j] * cost_points[i][j] 
                                           for i in range(num_segments_x+1) 
                                           for j in range(num_segments_y+1))
                           )
    opt_model.update()
    return None

In [9]:
def _add_nn_variables(opt_model: gp.Model, time_steps: int, nn_model: tf.keras.models.Sequential):

    nn_layer_data = {}
    nn_variables = {}

    for layer_num, layer in enumerate(nn_model.layers):

        layer_data = get_layer_data(layer)
        nn_layer_data[layer_num] = layer_data
        nn_variables[layer_num] = {}
        
        for time in range(time_steps):
            
            nn_variables[layer_num][time] = {"input_layer": opt_model.addMVar(layer_data.input_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_input"), 
                                            "affine_layer": opt_model.addMVar(layer_data.output_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_affine"),
                                            "output_layer": opt_model.addMVar(layer_data.output_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_output"),
                                            "penalty_cost": opt_model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, 
                                                                             name="Layer_"+str(layer_num)+"_T_"+str(time)+"_penalty_cost")
                                            }
    opt_model.update()
    return nn_layer_data, nn_variables

In [10]:
def _affine_transform(opt_model: gp.Model, layer_data: Layer_Data, nn_decision_variables: Dict[str, Any]):

    opt_model.addConstr(nn_decision_variables["affine_layer"] 
                        == nn_decision_variables["input_layer"] @ layer_data.coefficient_matrix + layer_data.bias_matrix)

    opt_model.update()
    
    return None

def _linear_func(opt_model: gp.Model, nn_decision_variables: Dict[str, Any]):

    opt_model.addConstr(nn_decision_variables["affine_layer"] 
                        == nn_decision_variables["output_layer"]) 
                        
    opt_model.addConstr(nn_decision_variables["penalty_cost"] == 0)

    opt_model.update()
    
    return None

def _proposed_relu_func(opt_model: gp.Model, nn_decision_variables: Dict[str, Any], output_size: int, penalty_value: float):

    opt_model.addConstr(nn_decision_variables["output_layer"] >= nn_decision_variables["affine_layer"])

    opt_model.addConstrs(nn_decision_variables["output_layer"][i] >= 0 
                         for i in range(output_size)
                        )

    opt_model.addConstr(nn_decision_variables["penalty_cost"] == 0 * penalty_value)

    opt_model.update()

    return None 

def _pctar_relu_func(opt_model: gp.Model, nn_decision_variables: Dict[str, Any], output_size: int, penalty_value: float):
    lower_bound = -10
    upper_bound = 10
    
    constant_1 = upper_bound / (upper_bound - lower_bound)
    constant_2 = lower_bound * constant_1
    
    opt_model.addConstr(nn_decision_variables["output_layer"] >= nn_decision_variables["affine_layer"])

    opt_model.addConstrs(nn_decision_variables["output_layer"][i] >= 0 
                         for i in range(output_size)
                        )

    opt_model.addConstrs(nn_decision_variables["output_layer"][i] 
                         <= constant_1 * nn_decision_variables["affine_layer"][i] - constant_2
                        for i in range(output_size)
                        )
    opt_model.addConstr(nn_decision_variables["penalty_cost"] 
                        == penalty_value * gp.quicksum(nn_decision_variables["output_layer"][i] 
                                                       for i in range(output_size)
                                                      )
                       )
    opt_model.update()
    return None

def _pcar_relu_func(opt_model: gp.Model, nn_decision_variables: Dict[str, Any], output_size: int, penalty_value: float):
    opt_model.addConstr(nn_decision_variables["output_layer"] >= nn_decision_variables["affine_layer"])
    opt_model.addConstrs(nn_decision_variables["output_layer"][i] >= 0 
                         for i in range(output_size)
                        )

    opt_model.addConstr(nn_decision_variables["penalty_cost"] 
                        == penalty_value * gp.quicksum(nn_decision_variables["output_layer"][i]
                                                       for i in range(output_size)
                                                      )
                       )
    opt_model.update()
    return None

In [11]:
def relu_nn_constraints(opt_model: gp.Model, nn_model: tf.keras.models.Sequential, 
                        relu_func: Callable[[gp.Model, Dict[str, Any], int], None], 
                        market_variables: Dict[str, Any], 
                        nn_scaler: Dict[str, Any], pwl_input: Bivariate_Pwl_Input, time_steps: int):
    
    nn_layer_data, nn_decision_variables = _add_nn_variables(opt_model, time_steps, nn_model)
    
    for time in range(time_steps):

        for layer_num, layer_data in nn_layer_data.items():
            # penalty_value = 10**(-layer_num)
            penalty_value = 10
            
            _affine_transform(opt_model, layer_data, nn_decision_variables[layer_num][time])

            if layer_data.activation_type == "relu":
                
                relu_func(opt_model, nn_decision_variables[layer_num][time], layer_data.output_size, penalty_value)
                
            else: 

                _linear_func(opt_model, nn_decision_variables[layer_num][time])
            
            if layer_num != list(nn_layer_data.keys())[-1]:  
                
                opt_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"] 
                                    == nn_decision_variables[layer_num + 1][time]["input_layer"])
                
        layer_num = list(nn_layer_data.keys())[0]

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][0] 
                            == ((market_variables[time]["flexibility_MWh"] - nn_scaler["x"]["min"]) 
                                / (nn_scaler["x"]["max"]- nn_scaler["x"]["min"])
                               )
                           )
        
        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][1] 
                            == ((market_variables[time]["updated_max_flexibility_MWh"] - nn_scaler["y"]["min"]) 
                                / (nn_scaler["y"]["max"] - nn_scaler["y"]["min"])
                               )
                           )

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][2] 
                            == ((pwl_input[time].shaping_parameters[0] - nn_scaler["a"]["min"]) 
                                / (nn_scaler["a"]["max"] - nn_scaler["a"]["min"])
                               )
                           )

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][3] 
                            == ((pwl_input[time].shaping_parameters[1] - nn_scaler["b"]["min"]) 
                                / (nn_scaler["b"]["max"] - nn_scaler["b"]["min"])
                               )
                           )
                            
        layer_num = list(nn_layer_data.keys())[-1]
                 
        opt_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"][0] 
                            == ((market_variables[time]["flexibility_cost_DKK"] - nn_scaler["z"]["min"]) 
                                / (nn_scaler["z"]["max"]- nn_scaler["z"]["min"])
                               )
                           )   

    opt_model.update()
    return nn_decision_variables

def gurobi_ml_model(opt_model: gp.Model, nn_model: tf.keras.models.Sequential, market_variables: Dict[str, Any], 
                    nn_variables: Dict[str, Any], pwl_data: Dict[str, Any], nn_scaler: Dict[str, Any], time_steps: int):
        
    layer_num_list = list(nn_variables.keys())

    for time in range(time_steps):

        layer_num = layer_num_list[0]
        opt_model.addConstr(nn_variables[layer_num][time]["input_layer"][0] 
                            == ((market_variables[time]["flexibility_MWh"] - nn_scaler["x"]["min"]) 
                                / (nn_scaler["x"]["max"]- nn_scaler["x"]["min"])
                               )
                           )
        
        opt_model.addConstr(nn_variables[layer_num][time]["input_layer"][1] 
                            == ((market_variables[time]["updated_max_flexibility_MWh"] - nn_scaler["y"]["min"]) 
                                / (nn_scaler["y"]["max"] - nn_scaler["y"]["min"])
                               )
                           )
    
        opt_model.addConstr(nn_variables[layer_num][time]["input_layer"][2] 
                            == ((pwl_data[time].shaping_parameters[0] - nn_scaler["a"]["min"]) 
                                / (nn_scaler["a"]["max"] - nn_scaler["a"]["min"])
                               )
                           )
    
        opt_model.addConstr(nn_variables[layer_num][time]["input_layer"][3] 
                            == ((pwl_data[time].shaping_parameters[1] - nn_scaler["b"]["min"]) 
                                / (nn_scaler["b"]["max"] - nn_scaler["b"]["min"])
                               )
                           )
                            
        layer_num = list(nn_layer_data.keys())[-1]
                 
        opt_model.addConstr(nn_variables[layer_num][time]["output_layer"][0] 
                            == ((market_variables[time]["flexibility_cost_DKK"] - nn_scaler["z"]["min"]) 
                                / (nn_scaler["z"]["max"]- nn_scaler["z"]["min"])
                               )
                           )    

    pred_constr = []
    for time in range(time_steps):
        input_vars = nn_variables[layer_num_list[0]][time]["input_layer"]
        output_vars = nn_variables[layer_num_list[-1]][time]["output_layer"]
        pred_constr.append(add_predictor_constr(opt_model, nn_model, input_vars, output_vars))

    opt_model.update()

    return pred_constr

In [12]:
def nn_objective_function(opt_model: gp.Model, market_variables: Dict[str, Any], nn_decision_variables: Dict[str, Any], 
                          market_data: Market_Data, time_steps: int):

    layer_num_list = list(nn_decision_variables.keys())
    opt_model.setObjective(gp.quicksum(market_data.market_price[time] * market_variables[time]["flexibility_MWh"] 
                                      - market_variables[time]["flexibility_cost_DKK"] for time in range(time_steps)) 
                           - gp.quicksum(nn_decision_variables[layer_num][time]["penalty_cost"] 
                                         for layer_num in layer_num_list for time in range(time_steps)), 
                           sense=GRB.MAXIMIZE)
    opt_model.update()
    return None

def new_nn_objective_function(opt_model: gp.Model, market_variables: Dict[str, Any], nn_decision_variables: Dict[str, Any], 
                              market_data: Market_Data, time_steps: int):

    layer_num_list = list(nn_decision_variables.keys())
    opt_model.setObjective(gp.quicksum(market_data.market_price[time] * market_variables[time]["flexibility_MWh"] 
                                      - 0.000001 * market_variables[time]["flexibility_cost_DKK"] for time in range(time_steps)) 
                           - gp.quicksum(nn_decision_variables[layer_num][time]["penalty_cost"] 
                                         for layer_num in layer_num_list for time in range(time_steps)), 
                           sense=GRB.MAXIMIZE)
    opt_model.update()
    return None

def pwl_objective_function(opt_model: gp.Model, market_variables: Dict[str, Any], market_data: Market_Data, time_steps: int):

    opt_model.setObjective(gp.quicksum(market_data.market_price[time] * market_variables[time]["flexibility_MWh"] 
                                      - market_variables[time]["flexibility_cost_DKK"] for time in range(time_steps)), 
                           sense=GRB.MAXIMIZE)
    opt_model.update()
    return None
    

In [13]:
def makedir(path):
    if not os.path.exists(path):
        os.mkdir(path)

width_results_path = os.path.join(cwd, "width results")
makedir(width_results_path)

length_results_path = os.path.join(cwd, "length results")
makedir(length_results_path)

folder_list = ["PWL", "Gurobi ML", "Proposed", "PCTAR", "PCAR", "FICNN"]

for folder in folder_list:
    makedir(os.path.join(width_results_path, folder))
    makedir(os.path.join(length_results_path, folder))

In [14]:
def get_results(pwl_data, market_data, market_variables, time_steps):
    
    variable_names = ["flexibility_MWh", "updated_max_flexibility_MWh", "flexibility_cost_DKK"]
    
    results_dict = {variable_name: np.array([market_variables[time][variable_name].X for time in range(time_steps)]) 
                    for variable_name in variable_names}

    results_dict["market_price"] = market_data.market_price 
    results_dict["shaping_parameters"] = [pwl_data[time].shaping_parameters for time in range(time_steps)]
    return pd.DataFrame(results_dict)
        

In [15]:
# Get the inputs for the optimization problem

input_path = os.path.join(cwd, "optimization input")
market_data_dict = read_pickle(os.path.join(input_path, "price_scenarios.pickle"))
pwl_data = read_pickle(os.path.join(input_path, "pwl_data_input.pickle"))
prosumer_cluster_data = read_pickle(os.path.join(input_path, "prosumer_cluster_data.pickle"))
nn_scaler = read_pickle(os.path.join(input_path, "nn_scaler.pickle"))
time_steps = 24
season = "Winter"
epsilon = 0.00

# Get NN model length paths
nn_length_path = os.path.join(cwd, "NN length models")
unn_model_length_path = os.path.join(nn_length_path, "Unconstrained model")
cnn_model_length_path = os.path.join(nn_length_path, "Constrained model")
ficnn_model_length_path = os.path.join(nn_length_path, "FICNN model")

# Get NN model length paths
nn_width_path = os.path.join(cwd, "NN width models")
unn_model_width_path = os.path.join(nn_width_path, "Unconstrained model")
cnn_model_width_path = os.path.join(nn_width_path, "Constrained model")
ficnn_model_width_path = os.path.join(nn_width_path, "FICNN model")
# Width names
width_list = ["half", "two", "five", "ten"]

# Length names
length_list = [str(i) for i in [1, 2, 4, 5, 6]]

columns = ["flexibility_MWh", "updated_max_flexibility_MWh", "flexibility_cost_DKK", "market_price", "shaping_parameters"]

In [ ]:
list(market_data_dict.keys())[2: 3]

## Gurobi machinelearning approach

In [ ]:
%%time

for width in width_list[2: 4]:
    unn_model = tf.keras.models.load_model(os.path.join(unn_model_width_path, width + "_Base_NN.keras"))
    results_dict = {}
    for scenario, market_data_list in list(market_data_dict.items())[2: 3]:
        results_dict[scenario] = {}
        for case, market_data in list(enumerate(market_data_list)):
            opt_model = gp.Model()
            market_variables = _add_market_variables(opt_model, time_steps)
            _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
            nn_layer_data, nn_variables = _add_nn_variables(opt_model, time_steps, unn_model)
        
            pred_constr = gurobi_ml_model(opt_model, unn_model, market_variables, nn_variables, pwl_data, nn_scaler, time_steps)
            nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
        
            # Set parameters before solving
            opt_model.setParam('TimeLimit', 3600)   # 60 seconds time limit
            opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
            # opt_model.setParam("Threads", 2)
    
            # Start measuring cpu time
            start_cpu_time = Time.process_time()
            
            # Optimize the model
        
            opt_model.optimize()
    
            # End measuring cpu time
            end_cpu_time = Time.process_time()
        
            # Save the results

            try:
                optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                            runtime=round(opt_model.Runtime, 3), 
                                                            cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                            mip_gap=opt_model.MIPGap)
            except AttributeError:
                if opt_model.status == GRB.TIME_LIMIT:
                    optimization_results = Optimization_Results(decision_variables=pd.DataFrame(),
                                                            runtime=round(opt_model.Runtime, 3), 
                                                            cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                            mip_gap=1000000)
    
            results_dict[scenario][str(case)] = optimization_results
            opt_model.dispose()
    write_pickle(results_dict, os.path.join(width_results_path, folder_list[1], width + "_Results.pickle"))

In [18]:
%%time

for length in length_list[-1: ]:
    unn_model = tf.keras.models.load_model(os.path.join(unn_model_length_path, length + "_layer_Base_NN.keras"))
    results_dict = {}
    for scenario, market_data_list in market_data_dict.items():
        results_dict[scenario] = {}
        for case, market_data in list(enumerate(market_data_list)):
            opt_model = gp.Model()
            market_variables = _add_market_variables(opt_model, time_steps)
            _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
            nn_layer_data, nn_variables = _add_nn_variables(opt_model, time_steps, unn_model)
        
            pred_constr = gurobi_ml_model(opt_model, unn_model, market_variables, nn_variables, pwl_data, nn_scaler, time_steps)
            nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
        
            # Set parameters before solving
            opt_model.setParam('TimeLimit', 3600)   # 60 seconds time limit
            opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
            # opt_model.setParam("Threads", 2)
    
            # Start measuring cpu time
            start_cpu_time = Time.process_time()
            
            # Optimize the model
        
            opt_model.optimize()
    
            # End measuring cpu time
            end_cpu_time = Time.process_time()
        
            # Save the results
            optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                        runtime=round(opt_model.Runtime, 3), 
                                                        cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                        mip_gap=opt_model.MIPGap)
    
            results_dict[scenario][str(case)] = optimization_results
            
    write_pickle(results_dict, os.path.join(length_results_path, folder_list[1], length + "_layer_Results.pickle"))
  

Set parameter TimeLimit to value 3600
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-10700T CPU @ 2.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1152 rows, 5184 columns and 8771 nonzeros
Model fingerprint: 0x7a0c5fa2
Model has 960 general constraints
Variable types: 5184 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-03, 1e+00]
  Objective range  [1e+00, 3e+02]
  Bounds range     [1e-01, 1e-01]
  RHS range        [2e-03, 1e+01]
Presolve removed 805 rows and 4856 columns
Presolve time: 0.08s
Presolved: 347 rows, 328 columns, 1585 nonzeros
Variable types: 285 continuous, 43 integer (43 binary)

Root relaxation: objective 1.431697e+04, 195 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl U

## Proposed method

In [ ]:
%%time

for width in width_list:
    cnn_model = tf.keras.models.load_model(os.path.join(cnn_model_width_path, width + "_Base_NN.keras"))    
    results_dict = {}
    for scenario, market_data_list in market_data_dict.items():
        results_dict[scenario] = {}
        for case, market_data in list(enumerate(market_data_list)):
            opt_model = gp.Model()
            market_variables = _add_market_variables(opt_model, time_steps)
            _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
            nn_variables = relu_nn_constraints(opt_model, cnn_model, _proposed_relu_func, market_variables, nn_scaler, pwl_data, time_steps)
            nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
        
        # Set parameters before solving
            opt_model.setParam('TimeLimit', 3600) # 3600 seconds time limit
            opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
        # Start measuring cpu time
            start_cpu_time = Time.process_time()
            
        # Optimize the model
            opt_model.optimize()
    
        # End measuring cpu time
            end_cpu_time = Time.process_time()
        
        # Save the results
    
            try:
                mip_gap = opt_model.MIPGap
            except AttributeError:
                if opt_model.STATUS == 2:
                    mip_gap = 0
                else:
                    mip_gap = "Unable to retrieve MIPGap"
            
            optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                        runtime=round(opt_model.Runtime, 3), 
                                                        cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                        mip_gap=mip_gap)
    
            results_dict[scenario][str(case)] = optimization_results
            
    write_pickle(results_dict, os.path.join(width_results_path, folder_list[2], width + "_Results.pickle"))


In [ ]:
%%time

for length in length_list[-1:]:
    cnn_model = tf.keras.models.load_model(os.path.join(cnn_model_length_path, length + "_layer_Base_NN.keras"))    
    results_dict = {}
    for scenario, market_data_list in market_data_dict.items():
        results_dict[scenario] = {}
        for case, market_data in list(enumerate(market_data_list)):
            opt_model = gp.Model()
            market_variables = _add_market_variables(opt_model, time_steps)
            _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
            nn_variables = relu_nn_constraints(opt_model, cnn_model, _proposed_relu_func, market_variables, nn_scaler, pwl_data, time_steps)
            nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
        
        # Set parameters before solving
            opt_model.setParam('TimeLimit', 3600) # 3600 seconds time limit
            opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
        # Start measuring cpu time
            start_cpu_time = Time.process_time()
            
        # Optimize the model
            opt_model.optimize()
    
        # End measuring cpu time
            end_cpu_time = Time.process_time()
        
        # Save the results
    
            try:
                mip_gap = opt_model.MIPGap
            except AttributeError:
                if opt_model.STATUS == 2:
                    mip_gap = 0
                else:
                    mip_gap = "Unable to retrieve MIPGap"
            
            optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                        runtime=round(opt_model.Runtime, 3), 
                                                        cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                        mip_gap=mip_gap)
    
            results_dict[scenario][str(case)] = optimization_results
            
    write_pickle(results_dict, os.path.join(length_results_path, folder_list[2], length + "_layer_Results.pickle"))


## FICNN Model

In [16]:
def _add_ficnn_variables(opt_model: gp.Model, time_steps: int, weights_dict):

    nn_layer_data = {}
    nn_variables = {}

    for layer_num, weights in weights_dict.items():
        nn_layer_data[layer_num] = {}
        nn_variables[layer_num] = {}
        for key, item in weights.items():
            nn_layer_data[layer_num][key] = item

        if "W_pos" not in weights.keys():
            nn_layer_data[layer_num]["input_size"] = nn_layer_data[layer_num]["W"][0].shape[0]
            nn_layer_data[layer_num]["output_size"] = nn_layer_data[layer_num]["W"][0].shape[1]
        else: 
            nn_layer_data[layer_num]["input_size"] = nn_layer_data[layer_num]["W_pos"][0].shape[0]
            nn_layer_data[layer_num]["output_size"] = nn_layer_data[layer_num]["W_pos"][0].shape[1]
        input_size = nn_layer_data[layer_num]["input_size"]
        output_size = nn_layer_data[layer_num]["output_size"]
        
        for time in range(time_steps):
            
            nn_variables[layer_num][time] = {"input_layer": opt_model.addMVar(input_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_input"), 
                                            "affine_layer": opt_model.addMVar(output_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_affine"),
                                            "output_layer": opt_model.addMVar(output_size, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, 
                                                                              name="Layer_"+str(layer_num)+"_T_"+str(time)+"_output"),
                                            "penalty_cost": opt_model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, 
                                                                             name="Layer_"+str(layer_num)+"_T_"+str(time)+"_penalty_cost")
                                            }
    opt_model.update()
    return nn_layer_data, nn_variables

def _affine_ficnn_transform(opt_model: gp.Model, layer_data, nn_decision_variables, nn_input_set):

    if "W_pos" not in layer_data.keys():
        opt_model.addConstr(nn_decision_variables["affine_layer"] 
                            == nn_decision_variables["input_layer"] @ layer_data["W"][0] + layer_data["W"][1])
    else:
        opt_model.addConstr(nn_decision_variables["affine_layer"] 
                            == nn_decision_variables["input_layer"] @ layer_data["W_pos"][0] + layer_data["W_pos"][1]
                               + nn_input_set @ layer_data["W"][0] + layer_data["W"][1])
    opt_model.update()
    
    return None

def _linear_func(opt_model: gp.Model, nn_decision_variables):

    opt_model.addConstr(nn_decision_variables["affine_layer"] 
                        == nn_decision_variables["output_layer"]) 
                        
    opt_model.addConstr(nn_decision_variables["penalty_cost"] == 0)

    opt_model.update()
    
    return None

def _relu_func(opt_model: gp.Model, nn_decision_variables, output_size: int, penalty_value: float):

    opt_model.addConstr(nn_decision_variables["output_layer"] >= nn_decision_variables["affine_layer"])

    opt_model.addConstrs(nn_decision_variables["output_layer"][i] >= 0 
                         for i in range(output_size)
                        )

    opt_model.addConstr(nn_decision_variables["penalty_cost"] == 0 * penalty_value)

    opt_model.update()

    return None 


def relu_ficnn_constraints(opt_model: gp.Model, weights_dict, 
                           market_variables: Dict[str, Any], nn_scaler: Dict[str, Any], 
                           pwl_input: Bivariate_Pwl_Input, time_steps: int):
    
    nn_layer_data, nn_decision_variables = _add_ficnn_variables(opt_model, time_steps, weights_dict)

    
    for time in range(time_steps):
        
        for layer_num, layer_data in nn_layer_data.items():
        
            output_size = layer_data["output_size"]
            nn_variable_set = nn_decision_variables[layer_num][time]
            
            _affine_ficnn_transform(opt_model, layer_data, nn_variable_set, nn_decision_variables["0"][time]["input_layer"])
    
            if layer_num != list(nn_layer_data.keys())[-1]:
                _relu_func(opt_model, nn_variable_set, output_size , 1)
            else:
                _linear_func(opt_model, nn_variable_set)
    
            if layer_num != list(nn_layer_data.keys())[-1]:  
                opt_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"] 
                                        == nn_decision_variables[str(int(layer_num) + 1)][time]["input_layer"])   
            
        layer_num = list(nn_layer_data.keys())[0]

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][0] 
                            == ((market_variables[time]["flexibility_MWh"] - nn_scaler["x"]["min"]) 
                                / (nn_scaler["x"]["max"]- nn_scaler["x"]["min"])
                               )
                           )
        
        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][1] 
                            == ((market_variables[time]["updated_max_flexibility_MWh"] - nn_scaler["y"]["min"]) 
                                / (nn_scaler["y"]["max"] - nn_scaler["y"]["min"])
                               )
                           )

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][2] 
                            == ((pwl_input[time].shaping_parameters[0] - nn_scaler["a"]["min"]) 
                                / (nn_scaler["a"]["max"] - nn_scaler["a"]["min"])
                               )
                           )

        opt_model.addConstr(nn_decision_variables[layer_num][time]["input_layer"][3] 
                            == ((pwl_input[time].shaping_parameters[1] - nn_scaler["b"]["min"]) 
                                / (nn_scaler["b"]["max"] - nn_scaler["b"]["min"])
                               )
                           )
                            
        layer_num = list(nn_layer_data.keys())[-1]
                 
        opt_model.addConstr(nn_decision_variables[layer_num][time]["output_layer"][0] 
                            == ((market_variables[time]["flexibility_cost_DKK"] - nn_scaler["z"]["min"]) 
                                / (nn_scaler["z"]["max"]- nn_scaler["z"]["min"])
                               )
                           )   

    opt_model.update()
    return nn_decision_variables


In [18]:
%%time

for width in width_list:
    weights_dict = read_pickle(os.path.join(ficnn_model_width_path, width + "_Base_NN.pickle"))    
    results_dict = {}
    for scenario, market_data_list in market_data_dict.items():
        results_dict[scenario] = {}
        for case, market_data in list(enumerate(market_data_list)):
            opt_model = gp.Model()
            market_variables = _add_market_variables(opt_model, time_steps)
            _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
            nn_variables = relu_ficnn_constraints(opt_model, weights_dict, market_variables, nn_scaler, pwl_data, time_steps)
            nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
        
        # Set parameters before solving
            opt_model.setParam('TimeLimit', 3600) # 3600 seconds time limit
            opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
        # Start measuring cpu time
            start_cpu_time = Time.process_time()
            
        # Optimize the model
            opt_model.optimize()
    
        # End measuring cpu time
            end_cpu_time = Time.process_time()
        
        # Save the results
    
            try:
                mip_gap = opt_model.MIPGap
            except AttributeError:
                if opt_model.STATUS == 2:
                    mip_gap = 0
                else:
                    mip_gap = "Unable to retrieve MIPGap"
            
            optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                        runtime=round(opt_model.Runtime, 3), 
                                                        cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                        mip_gap=mip_gap)
    
            results_dict[scenario][str(case)] = optimization_results
            
    write_pickle(results_dict, os.path.join(width_results_path, folder_list[5], width + "_Results.pickle"))


Set parameter Username
Academic license - for non-commercial use only - expires 2026-09-18
Set parameter TimeLimit to value 3600
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11.0 (22631.2))

CPU model: Intel(R) Core(TM) i7-10700T CPU @ 2.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 2232 rows, 1752 columns and 7835 nonzeros
Model fingerprint: 0x31600a29
Coefficient statistics:
  Matrix range     [3e-05, 1e+00]
  Objective range  [1e+00, 3e+02]
  Bounds range     [1e-01, 1e-01]
  RHS range        [5e-03, 1e+01]
Presolve removed 1932 rows and 1400 columns
Presolve time: 0.01s
Presolved: 300 rows, 352 columns, 2039 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
     220    1.4323390e+04   0.000000e+00   0.000000e+00      0s

Solved in 220 iterations an

In [23]:
%%time

for length in length_list:
    weights_dict = read_pickle(os.path.join(ficnn_model_length_path, length + "_layer_Base_NN.pickle"))    
    results_dict = {}
    for scenario, market_data_list in market_data_dict.items():
        results_dict[scenario] = {}
        for case, market_data in list(enumerate(market_data_list)):
            opt_model = gp.Model()
            market_variables = _add_market_variables(opt_model, time_steps)
            _add_market_constraints(opt_model, market_variables, prosumer_cluster_data[season], market_data.activation_signal, epsilon)
            nn_variables = relu_ficnn_constraints(opt_model, weights_dict, market_variables, nn_scaler, pwl_data, time_steps)
            nn_objective_function(opt_model, market_variables, nn_variables, market_data, time_steps)
            
        # Set parameters before solving
            opt_model.setParam('TimeLimit', 3600) # 3600 seconds time limit
            opt_model.setParam('MIPGap', 0.01)    # Stop when the relative gap is 1% (0.01)
        # Start measuring cpu time
            start_cpu_time = Time.process_time()
            
        # Optimize the model
            opt_model.optimize()
    
        # End measuring cpu time
            end_cpu_time = Time.process_time()
        
        # Save the results
    
            try:
                mip_gap = opt_model.MIPGap
            except AttributeError:
                if opt_model.STATUS == 2:
                    mip_gap = 0
                else:
                    mip_gap = "Unable to retrieve MIPGap"
            
            optimization_results = Optimization_Results(decision_variables=get_results(pwl_data, market_data, market_variables, time_steps),
                                                        runtime=round(opt_model.Runtime, 3), 
                                                        cpu_time=round(end_cpu_time - start_cpu_time, 3), 
                                                        mip_gap=mip_gap)
    
            results_dict[scenario][str(case)] = optimization_results
            
    write_pickle(results_dict, os.path.join(length_results_path, folder_list[5], length + "_layer_Results.pickle"))


Set parameter TimeLimit to value 3600
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11.0 (22631.2))

CPU model: Intel(R) Core(TM) i7-10700T CPU @ 2.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 4104 rows, 3144 columns and 11099 nonzeros
Model fingerprint: 0x43562136
Coefficient statistics:
  Matrix range     [4e-04, 2e+00]
  Objective range  [1e+00, 3e+02]
  Bounds range     [1e-01, 1e-01]
  RHS range        [6e-04, 1e+01]
Presolve removed 3764 rows and 2803 columns
Presolve time: 0.00s
Presolved: 340 rows, 341 columns, 1485 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
     249    1.4310692e+04   0.000000e+00   0.000000e+00      0s

Solved in 249 iterations and 0.02 seconds (0.01 work units)
Optimal objective  1.431069176e+04
Set parameter TimeLimi